### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="marketing_campaign",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/rodsaldanha/arketing-campaign",
    download_description="""
We download the data from the Kaggle repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/marketing_campaign/ && cd local-data-warehouse/marketing_campaign/ && kaggle datasets download rodsaldanha/arketing-campaign && cd ../../ && unzip local-data-warehouse/marketing_campaign/arketing-campaign.zip -d local-data-warehouse/marketing_campaign/ && rm local-data-warehouse/marketing_campaign/arketing-campaign.zip && rm local-data-warehouse/marketing_campaign/marketing_campaign.xlsx
""",
    # References
    academic_reference_bibtex=r"""@misc{saldanha2020marketing,
  author       = {Saldanha, Rodolfo},
  title        = {Marketing Campaign},
  year         = {2020},
  howpublished = {\url{https://www.kaggle.com/datasets/rodsaldanha/arketing-campaign}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="saldanha2020marketing",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We ensure Dt_Customer is a pandas datetime.
- We drop the ID column.
- We rename the values of the target variable to be more descriptive.
- We drop constant columns.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Response",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Response",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/marketing_campaign.csv", sep=";")
df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"], inplace=True)


cat_features = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
    "Marital_Status",
    "Education",
    "Response",
]

df[cat_features] = df[cat_features].astype("category")


df["Response"] = df["Response"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp1"] = df["AcceptedCmp1"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp2"] = df["AcceptedCmp2"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp3"] = df["AcceptedCmp3"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp4"] = df["AcceptedCmp4"].map({0: 'No', 1: 'Yes'})
df["AcceptedCmp5"] = df["AcceptedCmp5"].map({0: 'No', 1: 'Yes'})

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%Y-%m-%d")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 2,240
Columns: 26
Use sampling: False (sample size: 2,240)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Income', 'MntWines', 'Dt_Customer', 'MntMeatProducts', 'MntGoldProds', 'MntFishProducts', 'MntSweetProducts', 'MntFruits', 'Recency', 'Year_Birth']
Rows remaining as candidates after top-10 filter: 400 (of 2,240)

#### Duplicate Report
Total duplicate rows: 182 (8.12% of dataset)
Duplicate rows ignoring target: 201 (8.97% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response
0,1953,Graduation,Single,40464.0,0,1,2013-01-11,78,424,17,118,7,23,41,6,8,2,8,8,No,No,No,No,No,0,No
1,1960,Graduation,Widow,47916.0,0,1,2012-11-22,72,505,0,26,0,0,75,5,7,4,6,6,No,Yes,No,No,No,0,No
2,1972,Basic,Married,14188.0,0,0,2013-02-28,40,2,7,11,16,12,27,1,2,0,4,6,No,No,No,No,No,0,No
3,1969,Graduation,Together,76653.0,0,0,2013-08-16,91,736,63,946,219,189,126,1,4,7,11,2,No,No,Yes,Yes,No,0,No
4,1958,Graduation,Together,65196.0,0,2,2013-07-25,34,743,19,181,12,0,200,2,7,6,11,5,Yes,No,No,No,No,0,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Education,category,0.0,0.00,5.0,"Graduation, PhD, Master, 2n Cycle, Basic"
1,Marital_Status,category,0.0,0.00,8.0,"Married, Together, Single, Divorced, Widow, Alone, Absurd, YOLO"
2,AcceptedCmp3,category,0.0,0.00,2.0,"No, Yes"
3,AcceptedCmp4,category,0.0,0.00,2.0,"No, Yes"
4,AcceptedCmp5,category,0.0,0.00,2.0,"No, Yes"
5,AcceptedCmp1,category,0.0,0.00,2.0,"No, Yes"
6,AcceptedCmp2,category,0.0,0.00,2.0,"No, Yes"
7,Response,category,0.0,0.00,2.0,"No, Yes"
8,Dt_Customer,datetime64[ns],0.0,0.00,663.0,"2012-08-31 00:00:00, 2012-09-12 00:00:00, 2014-05-12 00:00:00, 2013-02-14 00:00:00, 2014-05-22 00:00:00, 2013-08-20 00:00:00, 2014-04-05 00:00:00, 2014-03-23 00:00:00, 2012-10-29 00:00:00, 2013-01-02 00:00:00"
9,Income,float64,24.0,1.07,1974.0,"7500.0, 35860.0, 63841.0, 34176.0, 39922.0, 83844.0, 48432.0, 80134.0, 67445.0, 46098.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Year_Birth,2240.0,1968.805804,11.984069,1893.0,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,666666.0
Kidhome,2240.0,0.444196,0.538398,0.0,2.0
Teenhome,2240.0,0.506250,0.544538,0.0,2.0
Recency,2240.0,49.109375,28.962453,0.0,99.0
MntWines,2240.0,303.935714,336.597393,0.0,1493.0
MntFruits,2240.0,26.302232,39.773434,0.0,199.0
MntMeatProducts,2240.0,166.950000,225.715373,0.0,1725.0
MntFishProducts,2240.0,37.525446,54.628979,0.0,259.0
MntSweetProducts,2240.0,27.062946,41.280498,0.0,263.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column         rank                                   
AcceptedCmp1   1                      No   2096  93.57
               2                     Yes    144   6.43
AcceptedCmp2   1                      No   2210  98.66
               2                     Yes     30   1.34
AcceptedCmp3   1                      No   2077  92.72
               2                     Yes    163   7.28
AcceptedCmp4   1                      No   2073  92.54
               2                     Yes    167   7.46
AcceptedCmp5   1                      No   2077  92.72
               2                     Yes    163   7.28
Dt_Customer    1     2012-08-31 00:00:00     12   0.54
               2     2012-09-12 00:00:00     11   0.49
               3     2014-05-12 00:00:00     11   0.49
               4     2013-02-14 00:00:00     11   0.49
               5     2014-05-22 00:00:00     10   0.45
Education      1              Graduation   1127  50.31
               2                     PhD    486  21.70
               3                  Master    370  16.52
               4                2n Cycle    203   9.06
               5                   Basic     54   2.41
Marital_Status 1                 Married    864  38.57
               2                Together    580  25.89
               3                  Single    480  21.43
               4                Divorced    232  10.36
               5                   Widow     77   3.44
Response       1                      No   1906  85.09
               2                     Yes    334  14.91

In [8]:
# Target Distribution
target_df

,count,pct
Response,,
No,1906,85.09
Yes,334,14.91


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to marketing_campaign/019d5cd5-f942-787a-8d75-a924527a671c
019d5cd5-f942-787a-8d75-a924527a671c
e06480e237baaf1282ba2d09d701e07455c9256b2169c2b37c790f119e361621
